# Output image to confirmed AI question

Run this notebook in a disposable JupyterLab project with a live Python kernel. It exports an existing stock notebook image, opens a preview, crops four pixels square, and optionally saves the derivative. The image reaches an AI question only after you click **Attach image**; confirmation does not run the question. The two AI questions need a configured provider. No device permission or personal media is required. This is a source-checkout example; PyPI 0.1.15 does not contain these tools.


## Make a disposable notebook output


In [ ]:
from io import BytesIO
import hashlib
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display, Image
from nbinlineai.tools import (list_outputs, export_output, open_media, close_media, crop_image, save_media, attach_media, release_media)
source_image = PILImage.new('RGB', (8, 8), (212, 64, 47))
encoded = BytesIO(); source_image.save(encoded, format='PNG')
PNG_BYTES = encoded.getvalue()
print('Disposable 8x8 output image ready; no file saved.')


In [ ]:
display(Image(data=PNG_BYTES))


## Export the existing output

Run the list call, then its inspection cell after the browser answers. Export its exact output ID and revision into owned memory. Neither step reruns the image cell or creates a file.


In [ ]:
listing = list_outputs('integration-output')
listing


In [ ]:
print(listing.status, listing.error)
output_ref = listing.result['outputs'][0] if listing.status == 'completed' else None
print('Output reference ready:', bool(output_ref))


In [ ]:
assert output_ref, 'Wait for list_outputs and rerun its inspection cell.'
exported = export_output(output_ref['cell_id'], output_ref['output_id'], output_ref['revision'])
exported


In [ ]:
print(exported.status, exported.error)
if exported.status == 'completed':
    assert exported.result.size == (8, 8)
    assert exported.result.getpixel((0, 0))[:3] == (212, 64, 47)
    print('Exported memory hash:', exported.media['sha256'])


## Preview the exported memory image

Opening a preview does not change the output or attach it to a question. Inspect its receipt in a later cell.


In [ ]:
assert exported.status == 'completed', 'Wait for export_output first.'
preview = open_media(exported.media)
preview


In [ ]:
print(preview.status, preview.result, preview.error)


## Make a new four-pixel-square derivative

The crop remains in owned memory and leaves the eight-pixel source unchanged. The later inspection checks actual decoded pixels and reports the exact encoded PNG hash.


In [ ]:
crop = crop_image(exported.media, 0, 0, 4, 4)
crop


In [ ]:
print(crop.status, crop.error)
if crop.status == 'completed':
    assert crop.result.size == (4, 4)
    assert crop.result.getpixel((0, 0))[:3] == (212, 64, 47)
    assert exported.result.size == (8, 8)
    print('Derivative memory hash:', crop.media['sha256'])


## Optional explicit save

Run the next call only if you want a separate file. It writes a new server-root-relative path and retains the memory derivative used below. The browser test exercises this optional step and removes the generated file after the question.


In [ ]:
assert crop.status == 'completed', 'Wait for crop_image first.'
saved = save_media(crop.media, save_to='auto')
saved


In [ ]:
print(saved.status, saved.media, saved.error)
if saved.status == 'completed':
    assert saved.media['sha256'] == crop.media['sha256']
    assert hashlib.sha256((Path.cwd() / saved.media['path']).read_bytes()).hexdigest() == crop.media['sha256']
    import json
    sidecar_path = Path.cwd() / saved.media['sidecar_path']
    assert sidecar_path.is_file()
    provenance = json.loads(sidecar_path.read_text())
    assert provenance['transform'] == 'crop_image'
    assert provenance['source_sha256'] == exported.media['sha256']
    assert provenance['output_sha256'] == crop.media['sha256']
    assert provenance['parameters'] == {'x': 0, 'y': 0, 'width': 4, 'height': 4}
    print('Saved path:', saved.media['path'])
    print('Sidecar path:', saved.media['sidecar_path'])


## Ask without an attachment

This first AI question has no image attached. It can discuss the notebook output as text context, but no native image input is sent merely because an output, preview, or derivative exists. Run it only after configuring a supported provider.


Before I confirm any image attachment, do you receive a native image from this notebook? Answer briefly.


## Confirm only the derivative for the next question

Run the call, click **Attach image** in the visible confirmation, then rerun the inspection cell until the receipt is complete. Confirmation changes only the identified question; it never runs it.


In [ ]:
assert crop.status == 'completed', 'Wait for the crop first.'
attached = attach_media(crop.media, 'integration-question', detail='auto')
attached


In [ ]:
print(attached.status, attached.result, attached.error)


## Preview context, then run the confirmed question

Open context preview to inspect its bounded estimate. Preview does not run this question. Click Run explicitly after the attachment indicator appears. The model receives the cropped image as a native image part, never as base64 tool text.


What is the visible color in the four-pixel-square image I explicitly attached?


## Clean up owned memory and optional file

Run after the question. The first cell closes the preview and releases managed bytes. The final cell removes only this example’s generated saved path, if the optional save ran.


In [ ]:
closed_preview = close_media(preview.result['preview_id']) if preview.status == 'completed' else None
released_export = release_media(exported.media['media_id']) if exported.status == 'completed' else None
released_crop = release_media(crop.media['media_id']) if crop.status == 'completed' else None
print('Cleanup requested; inspect the receipts later.')


In [ ]:
print('Preview:', closed_preview.status if closed_preview else 'none')
print('Export:', released_export.status if released_export else 'none')
print('Crop:', released_crop.status if released_crop else 'none')


In [ ]:
if 'saved' in globals() and saved.status == 'completed':
    saved_path = Path.cwd() / saved.media['path']
    assert saved_path.is_file()
    sidecar_path = Path.cwd() / saved.media['sidecar_path']
    assert sidecar_path.is_file()
    saved_path.unlink()
    sidecar_path.unlink()
    print('Optional generated file and sidecar removed:', not saved_path.exists() and not sidecar_path.exists())
else:
    print('No optional file was saved.')
